In [ ]:
import numpy as np
from skimage.util import view_as_windows
import os
import shutil
import gc
from tensorflow.data import Dataset
from tensorflow import TensorSpec
import h5py
from sklearn.utils import shuffle
from tensorflow.keras import layers,models
from tensorflow.data import AUTOTUNE
from tensorflow import optimizers
from tensorflow import losses
from tensorflow.keras import callbacks
from tensorflow.keras import backend

In [ ]:
def extract_patches(image, patch_size=(19, 19), stride=19):
    h, w, u = image.shape
    m, n = patch_size
    if m > h or n > w:
        raise ValueError("Patch size is larger than image dimensions")
    pad_h = m // 2
    pad_w = n // 2
    padded_image = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode='reflect')
    patches = view_as_windows(padded_image, (m, n, u), step=stride)
    patches = patches.reshape(-1, m, n, u)
    return patches


In [ ]:
imgs = [
    "/content/drive/MyDrive/hsi data/canola/canola_1.npy",
    "/content/drive/MyDrive/hsi data/canola/canola_2.npy",
    "/content/drive/MyDrive/hsi data/redroot/redroot_pigweed_1.npy",
    "/content/drive/MyDrive/hsi data/redroot/redroot_pigweed_2.npy",
    "/content/drive/MyDrive/hsi data/kochia/tmp/kochia_1.npy",
    "/content/drive/MyDrive/hsi data/kochia/tmp/kochia_2.npy",
    "/content/drive/MyDrive/hsi data/waterhemp/tmp/waterhemp_1.npy",
    "/content/drive/MyDrive/hsi data/waterhemp/tmp/waterhemp_2.npy",
    "/content/drive/MyDrive/hsi data/sugarbeet/sugarbeet_1.npy",
    "/content/drive/MyDrive/hsi data/sugarbeet/sugarbeet_2.npy",
    "/content/drive/MyDrive/hsi data/soybean/soybean_1.npy",
    "/content/drive/MyDrive/hsi data/soybean/soybean_2.npy",
    "/content/drive/MyDrive/hsi data/ragweed/ragweed_1.npy",
    "/content/drive/MyDrive/hsi data/ragweed/ragweed_2.npy",
    "/content/drive/MyDrive/hsi data/waterhemp/tmp/waterhemp_3.npy",
]

class_mapping = {
    "canola": 0,
    "kochia": 1,
    "ragweed": 2,
    "redroot": 3,
    "soybean": 4,
    "sugarbeet": 5,
    "waterhemp": 6,
}


hdf5_path = "/content/X_y_patches.h5"
n_samples = len(imgs)

with h5py.File(hdf5_path, "w") as f:
    for i, path in enumerate(imgs):
        img = np.load(path)
        patches = extract_patches(img)
        num_patches = patches.shape[0]
        patch_shape = patches.shape[1:]

        class_label = os.path.splitext(os.path.basename(path))[0].split("_")[0]
        class_id = class_mapping[class_label]
        print(patches.shape)
        f.create_dataset(f"X_{i}", data=patches, dtype="float32")
        f.create_dataset(f"y_{i}", data=np.array(class_id, dtype="int32"))



In [ ]:
# Датасеты
train_images = [0, 2, 4, 6, 8, 10, 12]
val_images = [1, 3, 5, 7, 9, 11, 13]
hdf5_path = "/content/X_y_patches.h5"

def make_generator(hdf5_path, image_indices):
    def generator():
        with h5py.File(hdf5_path, "r") as f:
            for idx in image_indices:
                patches = f[f"X_{idx}"][:]
                label = f[f"y_{idx}"][()]
                labels = np.full(patches.shape[0], label, dtype=np.int32)
                yield patches, labels
    return generator

dataset_train = Dataset.from_generator(
    make_generator(hdf5_path, train_images),
    output_signature=(
        TensorSpec(shape=(None, 19, 19, 224), dtype=np.float32),
        TensorSpec(shape=(None,), dtype=np.int32)
    )
).flat_map(lambda x, y: Dataset.from_tensor_slices((x, y))).shuffle(buffer_size=10000).batch(32).prefetch(AUTOTUNE)
dataset_val = Dataset.from_generator(
    make_generator(hdf5_path, val_images),
    output_signature=(
        TensorSpec(shape=(None, 19, 19, 224), dtype=np.float32),
        TensorSpec(shape=(None,), dtype=np.int32)
    )
).flat_map(lambda x, y: Dataset.from_tensor_slices((x, y))).shuffle(buffer_size=10000).batch(32).prefetch(AUTOTUNE)

In [ ]:
def build_model(input_shape = (19,19,224,1)):
  inp = layers.Input(input_shape)
  inp = layers.Reshape((19,19,224))(inp)
  conv1 = layers.Conv2D(128,kernel_size = (3,3),padding = "same")(inp)
  conv1 = layers.Activation("relu")(conv1)
  pool1 = layers.MaxPooling2D( pool_size = (2,2))(conv1)

  pool1 = layers.Reshape((9,9,128,1))(pool1)
  conv2 = layers.Conv3D(50,kernel_size = (1,1,8),strides = (1,1,5))(pool1)
  conv2 = layers.Activation("relu")(conv2)
  pool2 = layers.MaxPooling3D((2,2,1))(conv2)


  conv3 = layers.Conv3DTranspose(100,kernel_size=(3,3,39),padding = "same")(pool2)

  flat = layers.Flatten()(conv3)
  dense1 = layers.Dense(200,"relu")(flat)
  dense2 = layers.Dense(7,"softmax")(dense1)
  output = dense2
  model = models.Model(inp,output)
  return model


In [ ]:
model = build_model()
print(model.summary())

In [ ]:
opt = optimizers.Adam(learning_rate = 1e-4)
model.compile(optimizer=opt,loss = losses.SparseCategoricalCrossentropy(),metrics = ["accuracy"])

chechpoint = callbacks.ModelCheckpoint("/content/drive/MyDrive/hsi data/2d-3d_ver1.weights.h5",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only = True)
tb = callbacks.TensorBoard(
    log_dir="/content/logs",
    histogram_freq=1)

In [ ]:
history = model.fit(
    dataset_train,
    validation_data=dataset_val,
    epochs=5,
    callbacks=[chechpoint, tb]
)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'],markersize=20)
plt.plot(history.history['val_accuracy'], markersize=20)
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()
# plotting of training and validation loss curves
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
model.load_weights("/content/drive/MyDrive/hsi data/2d-3d_ver4.weights.h5")

In [ ]:
#model = models.load_model("/content/drive/MyDrive/hsi data/ver4.keras")

In [ ]:
def full_prediction(data_path,  image_indices):
  def make_test_generator(hdf5_path, image_indices):
    def generator():
        with h5py.File(hdf5_path, "r") as f:
            for idx in image_indices:
                patches = f[f"X_{idx}"][:]
                preds = model.predict(patches)
                yield preds
    return generator
  def create_aggregated_model():
    inputs = layers.Input(shape = (None,7))
    x = layers.GlobalAveragePooling1D()(inputs)
    outputs = layers.Activation("softmax")(x)
    model = models.Model(inputs,outputs)
    return model
  class_mapping = {
    "canola": 0, "kochia": 1, "ragweed": 2, "redroot": 3,
    "soybean": 4, "sugarbeet": 5, "waterhemp": 6
  }


  infmodel = create_aggregated_model()
  test_dataset = Dataset.from_generator(
      make_test_generator(data_path, image_indices),
      output_signature=TensorSpec(shape=(None, 7), dtype=np.float32)
  ).batch(1)
  final = infmodel.predict(test_dataset)
  predicted_classes = np.argmax(final, axis=1)
  predicted_labels = [list(class_mapping.keys())[c] for c in predicted_classes]
  print("Predicted classes:", predicted_labels)
  return predicted_labels

In [ ]:
p = full_prediction("/content/X_y_patches.h5",[14])